# Machine Learning 2025W — Exercise 1  
### Dataset Description and Exploration (heart disease)

**Group Members:**  
- Full Name 1  
- Full Name 2  
- Full Name 3  

---


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
import ssl
import certifi
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


ssl_context = ssl.create_default_context(cafile=certifi.where())

pd.set_option('display.max_columns', None)


## 2. Load Dataset

In [ ]:
# Fetch dataset by ID (45 = Heart Disease)
heart_disease = fetch_ucirepo(id=45)

# Split into features (X) and target (y)
X = heart_disease.data.features
y = heart_disease.data.targets

# Combine for convenience
df = pd.concat([X, y], axis=1)

print("Dataset shape:", df.shape)
df.head()

## 3. Basic Information

In [ ]:
df.info()
df.describe()
df.isna().sum()

## 4. Target Variable

In [ ]:
target_col = 'num' 
sns.countplot(x=target_col, data=df)
plt.title('Presence of heart disease (target)')
plt.xlabel('Heart Disease Presence (num)')
plt.show()


## 5. Feature Exploration

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols].hist(figsize=(12, 8))
plt.suptitle("Numeric Feature Distributions")
plt.show()

cat_cols = df.select_dtypes(exclude=np.number).columns
for col in cat_cols:
    plt.figure(figsize=(8, 3))
    sns.countplot(x=df[col])
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
corr = df.corr(numeric_only=True)['num'].sort_values(ascending=False)
corr

# Preprocessing

## Preprocess Target variable

In [ ]:
def target_bin(x):
    if x>0:
        return 1
    else:
        return 0

df["num"]= df["num"].apply(target_bin) # we binarize the target variable (1= presenze of a heart diseas, 0= not present)
df["diagnosis"]= df["num"] # rename target variable to "diagnosis"
df= df.drop(["num"], axis=1)
df.head()

### Preprocess Input variables

#### Check for outliers

In [ ]:
plt.figure()
plt.boxplot(df["age"])
plt.title("age") # no outliers

plt.figure()
plt.boxplot(df["trestbps"]) # expected range [90-200]
plt.title("trestbps") # no outliers

plt.figure()
plt.boxplot(df["chol"]) # expect range [100- max 400]
plt.title("chol") # here we can see some outliers which have to be handeled

plt.figure()
plt.boxplot(df["thalach"]) # expected range [80-200]
plt.title("thalach") # no clear outliers here

plt.figure()
plt.boxplot(df["oldpeak"]) # expected range [0-5]
plt.title("oldpeak") # we can see here some upper outliers which have to be handeld

plt.figure()
plt.boxplot(df["ca"].dropna()) # expected range [0-3]
plt.title("ca") # no outliers


In [ ]:
def handle_chol(x): #Cap values a 500 since more extreme values are very unlikly and would highly influence our model
    if x>500:
        return 500
    else:
        return x

def handle_oldpeak(x): #Cap values a 5.5 since more extreme values are very unlikly and would highly influence our model
    if x>5.5:
        return 5.5
    else:
        return x

df["chol"]= df["chol"].apply(handle_chol)
df["oldpeak"]= df["oldpeak"].apply(handle_oldpeak)

# check again visually for outliers
plt.figure()
plt.boxplot(df["chol"]) # expect range [100- max 400]
plt.title("chol") # here we can see some outliers which have to be handeled


plt.figure()
plt.boxplot(df["oldpeak"]) # expected range [0-5]
plt.title("oldpeak") # we can see here some upper outliers which have to be handeld

### Handle missing values

In [ ]:
contains_value = (df == '?').any().any()
contains_value #check if ther are any ? which could be potential null values

In [ ]:
pd.isna(df).any()


In [ ]:
df[pd.isna(df).any(axis=1)]

Here we can see that there are some missing values in the ca and thal columns. By inspecting the dataset we can see no logical meaningful reason for the NA values.

In [ ]:
print("Number of NA values in ca:")
print(df["ca"].isna().sum())
print("Number of NA values in thal:")
print(df["thal"].isna().sum())

We see that there are 4 missing values in the ca column and 2 missing values in the thal column. That are quite few, thats why we decided to just drop the rows with missing values.

In [ ]:
df=df.dropna(subset=["ca","thal"])
pd.isna(df).any()


### Split data into train and test sets

In [ ]:
X = df.drop("diagnosis", axis=1)  # features
y = df["diagnosis"]               # diagnosis

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,       # 20% of data for testing
    random_state=42,     # ensures reproducibility
    stratify=y           # keeps same class proportions in train/test
)

y_train.head()

### Making a scaled dataset

In [ ]:
X_train_scaled=X_train.copy()
X_test_scaled= X_test.copy()
scaler = StandardScaler()
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak'] # scale all numerical values exept ca
X_train_scaled[numeric_cols]= scaler.fit_transform(X_train_scaled[numeric_cols])
X_test_scaled[numeric_cols]= scaler.fit_transform(X_test_scaled[numeric_cols])
X_test_scaled.describe()

We now have scaled X in two new datasets all numerical values with a standard scaler such that mean is 0 and standard deviation is 1. WE have not included ca in the scalling since it only takes the values 0,1,2,3 and thus it would make a real difference to scale.

# Model Training

## Decision Tree Classifier

In [ ]:
# Create and train the model
clf = DecisionTreeClassifier(
    max_depth=5,           # Limit tree depth to prevent overfitting
    min_samples_split=20,   # Minimum samples required to split a node
    random_state=42
)
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)

In [ ]:
# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# View feature importance
# View feature importance
print("\nTop 10 most important features:")
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': clf.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(10))


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

# Define the parameter grid to search - Option 1 with aggressive pruning
param_grid = {
    'max_depth': [3, 5, 7],  # Try even shallower trees
    'min_samples_split': [30, 50, 100],  # More aggressive
    'min_samples_leaf': [10, 15, 20, 25],  # Larger leaves
    'ccp_alpha': [0.001, 0.005, 0.01, 0.02, 0.05],  # Pruning parameter
    'criterion': ['entropy']
}

# Create the model
clf = DecisionTreeClassifier(random_state=42)

# Setup GridSearchCV
grid_search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=5,                    # 5-fold cross-validation
    scoring='accuracy',      # or 'f1', 'roc_auc', etc.
    n_jobs=-1,              # Use all CPU cores
    verbose=2
)

# Fit the grid search
print("Starting grid search...")
grid_search.fit(X_train, y_train)

# Get the best parameters
print("\n" + "="*50)
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)
print("="*50)

# Use the best model
best_clf = grid_search.best_estimator_

# Predictions
train_pred = best_clf.predict(X_train)
y_pred = best_clf.predict(X_test)

# Evaluate
print("\nPerformance metrics:")
print(f"Train accuracy: {accuracy_score(y_train, train_pred):.4f}")
print(f"CV accuracy: {grid_search.best_score_:.4f}")
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Train-Test gap: {accuracy_score(y_train, train_pred) - accuracy_score(y_test, y_pred):.4f}")

In [ ]:
# 1. Check train accuracy
train_pred = best_clf.predict(X_train)
print("Train accuracy:", accuracy_score(y_train, train_pred))
print("CV accuracy:", 0.8137)
print("Test accuracy:", 0.7667)

# 2. Check class distribution
print("\nClass distribution in train:")
print(y_train.value_counts(normalize=True))
print("\nClass distribution in test:")
print(y_test.value_counts(normalize=True))

# 3. Look at detailed metrics
from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))